# ERD: entity-relationship diagrams of database schemas

`d3.erd()` draws tables with their columns, PK/FK/UK badges and crow's-foot cardinality from
a Mermaid `erDiagram` or a DataFrame of relationships. With `notebook=True` the interactive chart
is shown inline (isolated in an iframe): hover a table to highlight its relationships and
neighbours, search tables and columns, switch **Columns** between all, keys and none, or open the
side panels for layout, legend and export.

In [ ]:
import pandas as pd

from d3blocks import D3Blocks

# frame=False keeps node_properties as a dict, so entries can be edited by name below.
d3 = D3Blocks(frame=False, verbose="error")

## 1. Mermaid ER diagram

All four cardinality markers per end, `--` (identifying) and `..` (non-identifying) lines, and
entity blocks with `type name PK "comment"` rows are converted. Entities that only occur in a
relationship are drawn without columns.

In [ ]:
schema = """
erDiagram
    strategy_definitions ||--o{ strategy_runs : "has runs"
    strategy_definitions o|--o{ signals : produces
    strategy_runs o|--o{ signals : contains
    signals o|--o{ orders : "may produce"
    orders ||--o{ order_fills : "has fills"
    signals ||--o{ risk_check_results : "checked by"
    orders |o..o| broker_orders : "mirrored by"

    strategy_definitions {
        uuid id PK
        text name
        text strategy_type
        bool enabled
        timestamptz created_at
    }
    strategy_runs {
        uuid id PK
        uuid strategy_id FK
        text status
        timestamptz started_at
    }
    signals {
        uuid id PK
        uuid strategy_id FK "nullable"
        uuid run_id FK "nullable"
        text symbol
        text status
    }
    orders {
        uuid id PK
        uuid signal_id FK
        text broker_order_id UK
        numeric quantity
    }
    order_fills {
        uuid order_id PK,FK
        int fill_no PK
        numeric price
    }
"""
d3.erd(schema, notebook=True, filepath=None, figsize=[None, 650])

## 2. Keys only, top-down, with a captioned walkthrough

`show_columns='keys'` shows only PK/FK/UK columns (the top panel switches it at runtime).
`steps` highlight entities (`nodes`) and relationships (`edges`, in either order).

In [ ]:
steps = [
    {
        "title": "Strategies run",
        "caption": "A strategy definition has zero or more runs.",
        "highlight": {"edges": [("strategy_definitions", "strategy_runs")]},
    },
    {
        "title": "Runs emit signals",
        "caption": "Signals reference their strategy and run (both nullable).",
        "highlight": {
            "nodes": ["signals"],
            "edges": [
                ("strategy_runs", "signals"),
                ("strategy_definitions", "signals"),
            ],
        },
    },
    {
        "title": "Signals become orders",
        "caption": "An order is filled in one or more parts.",
        "highlight": {"edges": [("signals", "orders"), ("orders", "order_fills")]},
    },
]
d3.erd(
    schema,
    direction="TB",
    show_columns="keys",
    steps=steps,
    notebook=True,
    filepath=None,
    figsize=[None, 650],
)

## 3. Relationships from a DataFrame

`source_card` / `target_card` are `one`, `zero_or_one`, `one_or_more` or `zero_or_more`;
`identifying=False` draws a dashed line. Columns are passed per entity.

In [ ]:
rels = pd.DataFrame(
    {
        "source": ["customers", "orders", "products"],
        "target": ["orders", "order_lines", "order_lines"],
        "source_card": ["one", "one", "one"],
        "target_card": ["zero_or_more", "one_or_more", "zero_or_more"],
        "label": ["places", "contains", "ordered as"],
        "identifying": [True, True, False],
    }
)
entities = {
    "customers": {
        "columns": [
            {"name": "id", "type": "uuid", "key": "PK"},
            {"name": "email", "type": "text", "key": "UK"},
        ]
    },
    "orders": {
        "columns": [
            {"name": "id", "type": "uuid", "key": "PK"},
            {
                "name": "customer_id",
                "type": "uuid",
                "key": "FK",
                "comment": "who ordered",
            },
        ]
    },
    "order_lines": {
        "columns": [
            {"name": "order_id", "type": "uuid", "key": "PK,FK"},
            {"name": "product_id", "type": "uuid", "key": "PK,FK"},
            {"name": "quantity", "type": "int"},
        ]
    },
    "products": {
        "color": "#f28e2b",
        "columns": [{"name": "id", "type": "uuid", "key": "PK"}],
    },
}
d3.erd(rels, entities=entities, notebook=True, filepath=None, figsize=[None, 520])

## 4. Save as a standalone HTML file

Without `notebook=True` the chart is written to `filepath`; `showfig=True` opens it in the browser.

In [ ]:
d3.erd(schema, dark_mode="auto", filepath="erd_trading.html", showfig=False)
print("Saved", d3.config["filepath"])